# 2. Unified classical and TGIFS dynamics

Select a **saved** Fourier fit ID from `data/fits`; this notebook NEVER fits. Classical and TGIFS use the same model and optional cavity parameters.

In [ ]:
from pathlib import Path
from bosonic_algo import read_yaml, list_fits, run_or_load_classical, run_or_load_tgifs
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIT_ROOT = ROOT / "data" / "fits"
RESULT_ROOT = ROOT / "results"
config = read_yaml(ROOT / "configs" / "dynamics.yaml")
list_fits(FIT_ROOT)

## Explicitly choose a saved fit

This is intentionally not auto-selected to avoid using the wrong fitted Hamiltonian.

In [ ]:
FIT_ID = None  # e.g. "fit_0123456789abcdef"; pick from the printed fit table above
if not FIT_ID:
    raise ValueError("Choose an existing FIT_ID before propagation")

## Run or reuse classical dynamics

Exact and fitted classical results have separate cache identities. To avoid an expensive benchmark, comment out the exact call.

In [ ]:
exact, exact_meta, exact_hit = run_or_load_classical(config, potential="exact", result_root=RESULT_ROOT)
fitted, fitted_meta, fitted_hit = run_or_load_classical(config, potential="fitted", fit_id=FIT_ID, fit_root=FIT_ROOT, result_root=RESULT_ROOT)
print("exact reused:", exact_hit, "fitted reused:", fitted_hit)

## Run or reuse TGIFS

Set `cavity.enabled: true` in the YAML to add the photon qumode and dipole self-energy. Requires Qiskit and Bosonic-Qiskit.

In [ ]:
quantum, quantum_meta, quantum_hit = run_or_load_tgifs(FIT_ID, config, fit_root=FIT_ROOT, result_root=RESULT_ROOT)
print("TGIFS reused:", quantum_hit, "run ID:", quantum_meta["run_id"])

## Compare stored observables

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for key, panel in zip(("mean_r", "mean_R", "sigma_r", "sigma_R"), ax.ravel()):
    for name, data in (("exact CPU", exact), ("fitted CPU", fitted), ("TGIFS", quantum)):
        panel.plot(data["times"], data[key], label=name)
    panel.set_title(key)
ax[0, 0].legend()
plt.tight_layout()